## openaleph/follow the money (ftm): Who Owns This Block

For next Tuesday, we will be working in a collaborative openaleph enviroment, and investigating NYC property records.

I have set up our own instance of openaleph here:

https://aleph.floatingmedia.com

I have also set up individual user accounts for each of you--I will share the following via direct message in Slack:

user account: your columbia email
password: (probably change this)
APIkey: save this--you will need it (it's also accessible in "Setting" on openaleph)

We will be building individual investigations on openaleph, using the same workflow, datasets, ftm mappings, for the neighborhoods we choose.

Here is the essential workflow for this last week (steps 1-5) and this week (steps 6-10):

1. Choose a block (10 - 30) properties
2. Set up an investigation on our openaleph instance (shared with me: jt599@columbia.edu)
3. Obtain HPD building data (links and process provided)
4. Map all data into ftm ontology (process and yaml files provided)
5. Upload mapped data into your investigation via the openaleph api
6. Obtain ACRIS ownership data (links provided)
7. Obtain all deeds for your properties (link provided)
8. Upload deed pdfs to your investigation
9. Retrive the openaleph document ids from your investigation and merge with your ACRIS df
10. Repeat ftm mapping and uploading for ACRIS data (steps 4 & 5)

Feel free to use this notebook as a template, or to start a notebook on your own, and follow the workflow of the notebook.

Either way: carefully document, and outline your workflow in a single notebook for two reasons:
- you will want to submit the notebook as part of your homework
- you will want to be able to replicate the workflow potentially for a different neighborhood next week



### Step 6: Obtain ACRIS ownership deed data
OK, here are the challenge jumps up a bit. I am not providing code.

I will, however, provide:

- links to the property datasets
- the yaml file once you've gotten everything
- and some advice about both

#### ACRIS datasets

You will need all three of these data sets:

https://data.cityofnewyork.us/City-Government/ACRIS-Real-Property-Legals/8h5j-fqxa/about_data

https://data.cityofnewyork.us/City-Government/ACRIS-Real-Property-Master/bnx9-e6tj/about_data

https://data.cityofnewyork.us/City-Government/ACRIS-Real-Property-Parties/636b-3b5g/about_data

Important things to consider:

Make sure that you join the initial data set (legals) two at the very least the building ID and registration ID from the HPD dataset
You will want to do that on borough, block, and lot codes

You will then need to merge all of these, but make sure they stay as lean and duplicated as possible.

**Note** these are enormous files. I highly recommend using duckdb to accomplish this.

**Note 2** Document ID is the important key here.

**Note 3** make sure to filter your documents by **DEED** so that you are not also including other documents that will not be related to ownership.

At the end, you should have columns from all three of these, with connecting columns to HPD, and ideally, you should have only one row per building.

#### ACRIS YAML
It might help to look at the **acris.yml** to get a sense of what kind of data frame you're going to for this to map properly.

Read it carefully, and give some thought as to what a row would look like!

## ACRIS Data: Transforming and linking entities to the PDFs
Take you through some of the key elements of **Steps 6 through 10(*)**

**(*) Please note:** Above, I have changed the order of the final 5 steps (and added one) above because of a specific challenge in linking documents to entities that is built into openaleph's design.



In [ ]:
import pandas as pd
import duckdb

df_legal = pd.read_csv('data/real_property_legals.csv')
df_legal.head()
df_legal.columns

In [ ]:
df_harlem = pd.read_csv('data/harlem_block_registration.csv')
df_harlem.columns

In [ ]:
df_legal['BoroID'] = df_legal['BOROUGH']
df_legal['Block'] = df_legal['BLOCK']
df_legal['Lot'] = df_legal['LOT']

In [ ]:
df_harlem_legal = pd.merge(df_harlem, df_legal, on=['BoroID', 'Block', 'Lot'])
df_harlem_legal.columns

In [ ]:
df_master = pd.read_csv('data/real_property_master.csv')
df_master.head()

In [ ]:
df_harlem_legal_master = pd.merge(df_harlem_legal, df_master, on=['DOCUMENT ID'])
df_harlem_legal_master.head()

In [ ]:
import duckdb

In [ ]:
query = '''
SELECT *
FROM df_harlem_legal_master as df
LEFT JOIN read_csv_auto('data/real_property_parties.csv') as parties
ON df."DOCUMENT ID" = parties."DOCUMENT ID"
'''

df_merged = duckdb.sql(query).df()
df_merged.head()
# duckdb.sql(query).show()

In [ ]:
pd.set_option('display.max_columns', 70)

df_merged.head()


### End of Step 6: Merged Data
OK, to make this more accessible I am revealing at least my final data frame for the three merged ACRIS tables.



In [ ]:
# import pandas as pd

# pd.set_option('display.max_columns', 40)

# df_acris_final = pd.read_csv('flatbush_acris_finaID.csv')

In [ ]:
# df_acris_final

In [ ]:
df_merged.columns = df_merged.columns.str.lower().str.replace(' ', '_')
df_merged.columns = df_merged.columns.str.lower().str.replace('.', '_')
df_merged.columns = df_merged.columns.str.lower().str.replace('__', '_')

In [ ]:
df_merged_final = df_merged.loc[:, ~df_merged.columns.duplicated(keep='first')].copy()


In [ ]:
df_merged_final.shape

In [ ]:
df_merged_final['buildingid'].value_counts()

In [ ]:
df_merged_final['document_id'].nunique()

In [ ]:
df_merged_final[df_merged_final['document_id']=="FT_1650008590765"]

In [ ]:
df_merged_final['document_id'].value_counts()

In [ ]:
# df_acris_final.shape

In [ ]:
df_merged_final.to_csv('data/df_merged.csv', index=False)

### Step 7: Download PDFs of the Deeds

Using the documents ID from your neighborhood ACRIS dataset you can access PDFs of the deed via a link like this

https://a836-acris.nyc.gov/DS/DocumentSearch/DocumentImageView?doc_id=FT_3860000959686

This is 100% a playwright situation

Write a playwright script that goes to the page, downloads the complete PDF and name it consistently so that it has the document ID in the name!

In [ ]:
document_ids=df_merged_final['document_id'].astype(str).str.strip().unique().tolist()

In [ ]:
document_ids

In [ ]:
import asyncio
from pathlib import Path

from playwright.async_api import async_playwright, TimeoutError
from tqdm import tqdm

# ---------------------------------------------------------------------
# Your existing list
# document_ids = [...]
# ---------------------------------------------------------------------

DOWNLOAD_DIR = Path("downloads")
DOWNLOAD_DIR.mkdir(exist_ok=True)


async def download_document(page, doc_id):

    output_file = DOWNLOAD_DIR / f"{doc_id}.pdf"

    if output_file.exists():
        tqdm.write(f"✓ Skipping {doc_id}")
        return

    url = (
        "https://a836-acris.nyc.gov/DS/DocumentSearch/"
        f"DocumentImageView?doc_id={doc_id}"
    )

    tqdm.write(f"→ Processing {doc_id}")

    # ---------------------------------------------------------
    # Load page
    # ---------------------------------------------------------
    await page.goto(url)
    await page.wait_for_load_state("domcontentloaded")

    # ---------------------------------------------------------
    # FIX 1: Click SAVE correctly (PARENT td, not img)
    # ---------------------------------------------------------
    save_button = page.locator("td.vtm_buttonCell").filter(
        has=page.locator('img[title="Save"]')
    )

    await save_button.wait_for(state="visible", timeout=60000)
    await save_button.scroll_into_view_if_needed()
    await save_button.click(force=True)

    # ---------------------------------------------------------
    # Wait for dialog
    # ---------------------------------------------------------
    dialog = page.locator(".vtm_dialog")
    await dialog.wait_for(state="visible", timeout=60000)

    await page.wait_for_timeout(800)

    # ---------------------------------------------------------
    # Select "All"
    # ---------------------------------------------------------
    all_radio = dialog.locator(
        'input.vtm_exportDialogPageRangeRadio[value="0"]'
    )
    await all_radio.wait_for(state="visible", timeout=30000)
    await all_radio.click()

    # ---------------------------------------------------------
    # Trigger download (context-level listener = CRITICAL FIX)
    # ---------------------------------------------------------
    ok_button = dialog.locator("span.vtmBtn", has_text="OK")
    await ok_button.wait_for(state="visible", timeout=30000)

    download_task = page.context.wait_for_event(
        "download",
        timeout=240000
    )

    await page.wait_for_timeout(500)
    await ok_button.click(force=True)

    download = await download_task
    await download.save_as(output_file)

    tqdm.write(f"✓ Saved {doc_id}.pdf")


async def main():

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=False  # switch to True after confirming stability
        )

        context = await browser.new_context(
            accept_downloads=True
        )

        page = await context.new_page()

        for doc_id in tqdm(document_ids, desc="Downloading deeds"):

            for attempt in range(3):

                try:
                    await download_document(page, doc_id)
                    break

                except TimeoutError:
                    tqdm.write(f"⏳ Timeout {doc_id} (attempt {attempt+1}/3)")

                except Exception as e:
                    tqdm.write(f"❌ Error {doc_id}: {e}")

                if attempt < 2:
                    tqdm.write("Retrying in 5s...")
                    await page.wait_for_timeout(5000)

            else:
                tqdm.write(f"❌ FAILED permanently: {doc_id}")

        await browser.close()


# ---------------------------------------------------------
# Jupyter-safe run
# ---------------------------------------------------------
await main()

### Step 8: Upload the Deeds to your investigation!

And let's see how my instance of openaleph holds up!!! (Not wonderfully so far...)

I recommend just using the UI to upload, but here is an option via python

In [ ]:
#You could do this--or just use the interface
# from pathlib import Path
# import time
# COLLECTION_ID = '19'

# for doc in document_ids:
#     file_path = Path('acris_pdfs_flatbush/ACRIS_DEED_'+doc+'.pdf')
#     if file_path.exists():
#         size_in_mb = file_path.stat().st_size / (1024 * 1024)
#         if size_in_mb < 1:
#             print(f"Uploading: {file_path.name}...")
#             try:
#                 result=api.ingest_upload(
#                     collection_id=COLLECTION_ID, 
#                     file_path=file_path,  
#                     metadata={
#                         'file_name': file_path.name,
#                         'title': file_path.stem
#                     }
#                 )
#                 pdf_id_map[doc] = result.get('id')
#                 print(f"  → entity ID: {result.get('id')}")
#                 time.sleep(4)
#             except Exception as e:
#                 print(f"Failed to upload {file_path.name}: {e}")
#     else:
#         pdf_id_map[doc] = 'NotUploaded'
#         print(doc + ' no file')


# print("Batch upload complete!")

### Step 9: Retrive the openaleph document ids from your investigation and merge with your ACRIS df

This is a quirk of openaleph that I actually find quite odd, but you need to use the internal, aleph-generated document ID that is produced after your upload them, if you want to connect your documents to the entities.

Below I use **requests** to access the api.

In [ ]:
import os
import requests
ALEPH_KEY = os.environ.get("OPAL_API_KEY")


In [ ]:


HOST = "https://aleph.floatingmedia.com"
COLLECTION_ID='19'

headers = {"Authorization": f"ApiKey {ALEPH_KEY}"}

response = requests.get(
    f"{HOST}/api/2/entities",
    headers=headers,
    params={"filter:collection_id": COLLECTION_ID, "filter:schema": "Pages",
               "limit": 50,
    "offset": 0}
)
print(response.status_code)


In [ ]:
print(response.json())

In [ ]:
pdf_id_map = {}
num=1
docs_info = response.json()
for doc in docs_info['results']:
    print(doc['id'])
    doc_id = doc['properties']['fileName'][0].split("DEED_")[-1].replace(".pdf","")
    print(doc_id)
    print(num)
    pdf_id_map[int(doc_id)] = doc['id']
    num+=1

In [ ]:
pdf_id_map

In [ ]:
df_acris_final

In [ ]:
df_acris_final['pdf_entity_id'] = df_acris_final['document_id'].map(pdf_id_map)

In [ ]:
df_acris_final

In [ ]:
df_acris_final.to_csv('acris_doc_ids.csv', index=False)

### Step 10: map to ftm using an version of acris.yml
